# Qwen3.5-0.8B 日本語強化学習 (QLoRA / Unsloth)

本ノートブックは **Qwen3.5-0.8B (FP16)** を **llm-jp/oasst1-21k-ja** から抽出した
高品質な日本語指示データ 8,000 件を用いて **QLoRA** 微調整し、
日本語性能を底上げしつつ **破滅的忘却 (catastrophic forgetting)** を最小限に抑えることを目的とします。

## 設計の要点 (忘却抑制)

| 項目 | 設定 | 忘却抑制への寄与 |
|------|------|------------------|
| 手法 | QLoRA (4bit) | 元の重みを凍結し、低ランク差分のみ学習するため原本知識が保持される |
| ターゲット層 | `q/k/v/o/gate/up/down_proj` (全 attn + 全 MLP) | 広く適応させつつ、ランク 16 で表現力を制限 |
| LoRA Rank / Alpha | 16 / 32 (`alpha = 2r`) | ランクを控えめに保ち、過学習を防止 |
| LR | `2e-4` (cosine + warmup) | LoRA 向きの中程度の学習率。フル FT に比べ大幅に低い |
| Epochs | 3 | 過学習・忘却を防ぐため短めに設定 |
| データ | oasst1-21k-ja から高品質 8,000 件 | 多様性と品質を両立させ、特定パターンへの偏りを回避 |
| Optimizer | `paged_adamw_8bit` | VRAM 節約しつつ Adam 系の安定した収束 |
| Gradient Checkpointing | 有効 (Unsloth) | 長文脈 (2048) 学習を Colab T4 でも実行可能 |

> **備考**: モデル名 `Qwen/Qwen3.5-0.8B` は仕様書に基づく設定です。
> 公開時に同名モデルが存在しない場合は、`Qwen/Qwen3-0.6B` や `Qwen/Qwen2.5-1.5B`
> などの代替に差し替えてください (Unsloth は Qwen 系全モデルをサポート)。


## 1. 環境セットアップ

Unsloth は T4 / L4 / A100 / V100 すべてで動作します。
Colab の GPU ランタイム (T4 以上推奨) を選択してから実行してください。

In [ ]:
# GPU 確認 (T4 16GB 以上を想定)
!nvidia-smi


In [ ]:
# Unsloth インストール (Colab nightly 推奨)
# 参考: https://github.com/unslothai/unsloth
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir --no-deps "trl<0.9.0" peft accelerate bitsandbytes
# xformers は Unsloth が自動選択するが、Colab 環境では明示的に入れておくと安定
!pip install -q xformers==0.0.23.post1 triton==2.2.0


In [ ]:
# インポート & コンフィグ
import os
import random
import torch
import numpy as np
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments

# 再現性
SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ===== ハイパーパラメータ (仕様書準拠) =====
MODEL_NAME      = "Qwen/Qwen3.5-0.8B"     # ※存在確認が必要。無ければ Qwen/Qwen3-0.6B 等へ
MAX_SEQ_LEN     = 2048
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]
LR              = 2e-4
EPOCHS          = 3
GRAD_ACCUM      = 4
BATCH_SIZE      = 2           # T4 16GB でも 2048 は厳しめ。OOM 時は 1 へ
WARMUP_STEPS    = 50
DATASET_TARGET  = 8000        # oasst1-21k-ja から抽出する件数
OUTPUT_DIR      = "outputs"
ADAPTER_DIR     = "lora_adapter"

print(f"bf16 supported: {is_bfloat16_supported()}")
print(f"Model: {MODEL_NAME}")
print(f"LoRA r={LORA_R} alpha={LORA_ALPHA} dropout={LORA_DROPOUT}")
print(f"LR={LR} epochs={EPOCHS} grad_accum={GRAD_ACCUM} bs={BATCH_SIZE}")


## 2. モデル読み込み (Unsloth + 4bit QLoRA)

Unsloth の `FastLanguageModel.from_pretrained` を使い、
FP16 重みを 4bit 量子化して読み込みます (QLoRA)。
`dtype=None` を指定すると GPU に最適な型を Unsloth が自動選択します。

In [ ]:
# Qwen3.5-0.8B を 4bit 量子化でロード
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = torch.float16,   # 仕様: FP16
    load_in_4bit    = True,            # QLoRA
)
print(f"\\n[Loaded] vocab={tokenizer.vocab_size}, hidden={model.config.hidden_size}")


In [ ]:
# 念のためチャットテンプレートが存在するか確認
# 無ければ Qwen 標準 ChatML を設定
if tokenizer.chat_template is None:
    from unsloth.chat_templates import get_chat_template
    tokenizer = get_chat_template(
        tokenizer,
        chat_template = "chatml",   # Qwen 系標準
        mapping = {"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
    )
    print("chat_template を ChatML として設定しました")
else:
    print("既存 chat_template を使用します")


## 3. LoRA 適用 (ターゲット層: attn 全部 + MLP 全部)

仕様書の通り `q/k/v/o/gate/up/down` の 7 モジュールすべてを LoRA ターゲットにします。
これにより Attention と MLP の両方を適応させ、日本語表現の微調整を広く行います。
ランク 16 と `alpha = 2r = 32` の組み合わせは、強すぎず弱すぎない標準的な設定です。

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                = LORA_R,
    target_modules   = TARGET_MODULES,
    lora_alpha       = LORA_ALPHA,
    lora_dropout     = LORA_DROPOUT,
    bias             = "none",
    use_gradient_checkpointing = "unsloth",   # 長文脈学習に必須
    random_state     = SEED,
    use_rslora       = False,    # 通常 LoRA。rsLoRA は rank が大きい場合に有用
    loftq_config     = None,
)

# 学習可能パラメータ数を表示 (フル FT ではなくパラメータ効率の良さを確認)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


## 4. データセット準備 (llm-jp/oasst1-21k-ja → 高品質 8,000 件)

### 抽出方針
1. **言語**: 日本語エントリのみ (lang == "Japanese" または同等)
2. **品質**: oasst1 の `rank` フィールド (低いほど高品質) が存在する場合は rank ≤ 1 で絞る
3. **長さ**: 指示 30 文字以上 / 応答 50〜4000 文字 で極端に短い/長いノイズを除外
4. **多様性**: シャッフル後 8,000 件をサンプリング (特定スレッドへの偏りを回避)
5. **対話形式**: user / assistant の対を ChatML で成形し、EOS まで含めて学習

> 破滅的忘却の観点から、**多様性の確保**が極めて重要です。
> 同一トピック・同一話者への偏りを防ぐため、親スレッド ID で重複排除した上でサンプリングします。

In [ ]:
# データセット読み込み
raw = load_dataset("llm-jp/oasst1-21k-ja", split="train")
print(f"Total rows: {len(raw)}")
print("Columns:", raw.column_names)
print("\n--- Sample ---")
print(raw[0])


In [ ]:
# 構造に応じて正規化 (oasst1 系は message tree 形式が多い)
# 想定フィールド: text / role / lang / rank / parent_id / message_id など
# もし Alpaca 形式 (instruction/output) の場合は別ルートで処理

def _safe_get(d, *keys, default=None):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return default

# 1) アシスタント応答のみを取り出す (rank が低いほど高品質)
def is_high_quality_assistant(ex):
    role = _safe_get(ex, "role", "agent", default="")
    if isinstance(role, str) and role.lower() not in ("assistant", "agent"):
        return False
    text = _safe_get(ex, "text", "output", "content", default="")
    if not text or len(str(text)) < 50:
        return False
    rank = _safe_get(ex, "rank", "quality", default=None)
    if rank is not None:
        try:
            if float(rank) > 1.0:
                return False
        except (TypeError, ValueError):
            pass
    return True

filtered = raw.filter(is_high_quality_assistant)
print(f"After quality filter: {len(filtered)}")


In [ ]:
# 2) 長さフィルタ (極端値除去)
def length_filter(ex):
    text = _safe_get(ex, "text", "output", "content", default="")
    n = len(str(text))
    return 50 <= n <= 4000

filtered = filtered.filter(length_filter)
print(f"After length filter: {len(filtered)}")


In [ ]:
# 3) 親スレッドで重複排除 (同一スレッドの応答を 1 件に絞ることで多様性を確保)
parent_key = "parent_id" if "parent_id" in filtered.column_names else None
if parent_key:
    seen = set()
    def dedup(ex):
        pid = ex[parent_key]
        if pid in seen:
            return False
        seen.add(pid)
        return True
    filtered = filtered.filter(dedup)
    print(f"After dedup by {parent_key}: {len(filtered)}")
else:
    print("parent_id 無し -> スキップ")


In [ ]:
# 4) シャッフル & 8,000 件サンプリング
filtered = filtered.shuffle(seed=SEED)
N = min(DATASET_TARGET, len(filtered))
sampled = filtered.select(range(N))
print(f"Final sampled: {len(sampled)}")


In [ ]:
# 5) 各アシスタント応答に対応する user 入力を復元して (prompt, response) 対を作る
# oasst1 は木構造なので、parent_id を辿って user 発話を取得
# 簡略化: データセットに user メッセージも含まれている前提で ID マップを構築

id2msg = {ex["message_id"]: ex for ex in raw if "message_id" in ex}

def build_pair(assistant_ex):
    # 1 つ上の親 (user 発話) を取得
    pid = assistant_ex.get("parent_id")
    if pid and pid in id2msg:
        parent = id2msg[pid]
        user_text = _safe_get(parent, "text", "content", default="")
    else:
        user_text = ""
    assistant_text = _safe_get(assistant_ex, "text", "output", "content", default="")
    return {"user": str(user_text), "assistant": str(assistant_text)}

pairs = [build_pair(ex) for ex in sampled]
pairs = [p for p in pairs if p["user"] and p["assistant"]]
print(f"Built pairs: {len(pairs)}")
print("\n--- Example pair ---")
print("USER:", pairs[0]["user"][:200])
print("ASSISTANT:", pairs[0]["assistant"][:200])


In [ ]:
# 6) ChatML 形式でトークン列を構築
def to_chat_text(p):
    msgs = [
        {"role": "user",      "content": p["user"]},
        {"role": "assistant", "content": p["assistant"]},
    ]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return text

texts = [to_chat_text(p) for p in pairs]
print("=== Sample formatted text ===")
print(texts[0][:600])
print("\n=== Token length stats ===")
import numpy as np
lens = [len(tokenizer.encode(t, add_special_tokens=False)) for t in texts[:500]]
print(f"min={min(lens)} p50={int(np.percentile(lens,50))} p95={int(np.percentile(lens,95))} max={max(lens)}")


In [ ]:
# 7) Dataset オブジェクト化
train_ds = Dataset.from_dict({"text": texts})
print(train_ds)


## 5. 学習

`paged_adamw_8bit` オプティマイザで VRAM を節約しつつ、
cosine スケジュール + warmup で学習を安定化させます。
`save_strategy="epoch"` で各 epoch のアダプタを保存し、
最良 epoch を後で選べるようにします。

In [ ]:
# TrainingArguments (仕様書準拠)
training_args = TrainingArguments(
    output_dir              = OUTPUT_DIR,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    warmup_steps            = WARMUP_STEPS,
    num_train_epochs        = EPOCHS,
    learning_rate           = LR,
    fp16                    = not is_bfloat16_supported(),
    bf16                    = is_bfloat16_supported(),
    logging_steps           = 10,
    optim                   = "paged_adamw_8bit",
    weight_decay            = 0.01,
    lr_scheduler_type       = "cosine",
    seed                    = SEED,
    save_strategy           = "epoch",
    save_total_limit        = EPOCHS,
    report_to               = "none",     # wandb 等を使う場合は "wandb"
    max_grad_norm           = 1.0,
    gradient_checkpointing  = False,      # Unsloth 側で既に有効化済み
)

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = train_ds,
    dataset_text_field = "text",
    max_seq_length   = MAX_SEQ_LEN,
    args             = training_args,
    packing          = False,    # 短い対話が多い場合は True で効率化も可
)

trainer_stats = trainer.train()
print(f"\\nTotal steps: {trainer_stats.global_step}")
print(f"Train loss (final): {trainer_stats.training_loss:.4f}")


In [ ]:
# 学習曲線を簡易表示
import pandas as pd
log_df = pd.DataFrame(trainer.state.log_history)
log_df = log_df[log_df["loss"].notna()]
print(log_df.tail(10))


## 6. 保存

LoRA アダプタのみ保存し、マージ済みモデルは別途保存します。
Colab の Google Drive に退避する場合は、右側フォルダアイコンからマウントしてください。

In [ ]:
# LoRA アダプタのみ (軽量)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved adapter to {ADAPTER_DIR}/")


In [ ]:
# (任意) 16bit でマージして保存 (推論用・HF Hub アップロード用)
# Colab の無料 T4 でも 0.8B なら 16bit マージは問題なく動作します
MERGED_DIR = "merged_model_fp16"
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
print(f"Saved merged 16bit model to {MERGED_DIR}/")


## 7. 推論テスト (日本語 + 英語での忘却確認)

破滅的忘却が起きていないか確認するため、
- 日本語の指示追従性 (向上しているべき)
- 英語での基本的な質問応答 (低下していないべき)

の両方をサンプリングして比較します。

In [ ]:
FastLanguageModel.for_inference(model)  # 推論モード

def ask(prompt, max_new_tokens=256):
    msgs = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    return decoded.strip()

# 日本語テスト
jp_prompts = [
    "日本の四季について短く説明してください。",
    "Python のリスト内包表記の利点を教えてください。",
    "健康的な朝食のアイデアを 3 つ提案してください。",
]
print("=== 日本語推論 ===")
for p in jp_prompts:
    print(f"\n[Q] {p}")
    print(f"[A] {ask(p)}")


In [ ]:
# 英語テスト (忘却チェック)
en_prompts = [
    "Explain the difference between TCP and UDP in one paragraph.",
    "What is the capital of France?",
    "Write a haiku about the ocean.",
]
print("=== English inference (forgetting check) ===")
for p in en_prompts:
    print(f"\n[Q] {p}")
    print(f"[A] {ask(p)}")


## 8. (任意) Hugging Face Hub へアップロード

学習したアダプタを公開・共有する場合の例です。
Colab シークレットに `HF_TOKEN` を登録してからコメントアウトを外してください。

In [ ]:
# from huggingface_hub import login
# from google.colab import userdata
# login(userdata.get("HF_TOKEN"))
#
# HF_REPO = "your-username/qwen3.5-0.8b-ja-qlora-oasst1-8k"
# model.push_to_hub_merged(HF_REPO, tokenizer, save_method="merged_16bit")
# model.push_to_hub(HF_REPO)  # adapter のみ


## 9. 今後の改善方向

1. **評価の定量化和**:
   - `llm-jp-eval` / `jp-eval` で日本語タスクのスコアを比較
   - `lm-evaluation-harness` で MMLU / MMLU-JA を実行
2. **混合データによる忘却抑制の強化**:
   - 英語汎用データ (例: 1k 件の多言語 Wikipedia) を 10% ブレンドすると更に安定
3. **対話継続性能の強化**:
   - oasst1 は単発対話中心なので、マルチターンデータ (`kunishou/databricks-dolly-15k-ja` 等) を追加
4. **DPO / ORPO による好みチューニング**:
   - 本 SFT のあとに DPO を載せると、応答の自然さが向上する場合があります
5. **ランキング戦略の改善**:
   - rank だけでなく、ペアレントスレッドのトピック多様性をクラスタリングで担保

詳細は `ROADMAP.md` を参照してください。
